# pricetaker walkthrough

A narrated, end-to-end tour of the battery price-arbitrage simulator on the
bundled **synthetic** sample data (no ENTSO-E token or network needed).

We will:
1. Load day-ahead prices.
2. Define a battery.
3. Run several dispatch strategies.
4. Compare their economics and visualise the dispatch.

> Reminder: this is a teaching tool, not financial advice. The best-case
> strategy assumes perfect price foresight.

In [ ]:
from pricetaker import plotting
from pricetaker.battery import BatteryParams
from pricetaker.data.sample import load_sample
from pricetaker.metrics import compare
from pricetaker.strategies import DailyTopN, PerfectForesight, RollingHorizon, Threshold

prices = load_sample("AT")  # hourly, synthetic, EUR/MWh
prices.head()

The prices are a tz-aware `pandas.Series`. Note the daily shape: cheap at
night and midday (solar), expensive in the evening peak.

In [ ]:
prices.plot(figsize=(11, 3), title="AT day-ahead prices (synthetic)");

## Define a battery

A 10 MW / 20 MWh battery (2-hour duration) with an 86% round-trip efficiency.

In [ ]:
battery = BatteryParams(power_mw=10, energy_mwh=20, eta_rt=0.86)
print(f"one-way efficiency each side: {battery.eta_charge:.3f}")
print(f"break-even price ratio (1/eta_rt): {1 / battery.eta_rt:.3f}")

So the high price must be ~16% above the low price before a cycle even breaks
even. Cheap swings are not worth capturing.

## Run the strategies

In [ ]:
dispatches = {
    "perfect_foresight": PerfectForesight().run(prices, battery),
    "rolling_horizon": RollingHorizon(horizon_hours=48).run(prices, battery),
    "daily_topn": DailyTopN(n=4).run(prices, battery),
    "threshold": Threshold(low_pct=25, high_pct=75).run(prices, battery),
}
compare(dispatches)[["net_profit_eur", "equivalent_full_cycles", "spread_captured"]]

Perfect foresight is the ceiling; the heuristics sit below it. The gap from
`rolling_horizon` up to `perfect_foresight` is the value of a better forecast.

In [ ]:
plotting.strategy_comparison(dispatches).show()

## Inspect a single dispatch

Watch the LP buy in the cheap/negative troughs and sell into the peaks.

In [ ]:
lp = dispatches["perfect_foresight"]
plotting.price_and_dispatch(lp).show()
plotting.state_of_charge(lp).show()
plotting.cumulative_profit(lp).show()

## Negative prices

When the price dips below zero, the optimiser is *paid* to charge. Count how
often that happens in the sample and how much energy the LP soaks up then.

In [ ]:
negative = lp.prices < 0
print(f"negative-price steps: {int(negative.sum())} of {len(lp.prices)}")
print(f"charging during negative prices: {lp.charge_mw[negative].sum() * lp.dt_hours:.1f} MWh")

## Try it yourself

- Raise `deg_cost_eur_mwh` and watch marginal cycles disappear.
- Load `load_sample("DE_LU")` (15-minute resolution) — everything still works.
- Tighten `eta_rt` and see the break-even spread widen.